In [1]:
!pip install -q fastapi uvicorn pyngrok pillow transformers accelerate
!pip install -q 'git+https://github.com/allenai/lerobot.git@molmoact2-policy#egg=lerobot[molmoact2]'
!pip install -q 'av>=12.0.0' --upgrade

DEPRECATION: git+https://github.com/allenai/lerobot.git@molmoact2-policy#egg=lerobot[molmoact2] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Patch pyav_utils.py so av.option.Option annotation is not evaluated at import time
import glob
for path in glob.glob('/usr/local/lib/python3.*/dist-packages/lerobot/datasets/pyav_utils.py'):
    src = open(path).read()
    if 'from __future__ import annotations' not in src:
        open(path, 'w').write('from __future__ import annotations\n' + src)
        print(f'Patched: {path}')
    else:
        print(f'Already patched: {path}')

Already patched: /usr/local/lib/python3.13/dist-packages/lerobot/datasets/pyav_utils.py


In [3]:
import os
import threading
import time
import types
import numpy as np
import torch
import torchvision.transforms.functional as TVF
from google.colab import userdata
from pyngrok import ngrok
from huggingface_hub import login
from transformers import AutoProcessor

# ── Secrets ───────────────────────────────────────────────────────────────────
os.environ['HF_TOKEN']    = userdata.get('HF_TOKEN')
os.environ['NGROK_TOKEN'] = userdata.get('NGROK_TOKEN')

login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF login OK')

# ── Load fine-tuned policy ────────────────────────────────────────────────────
from lerobot.policies.molmoact2 import MolmoAct2Policy

REPO_ID    = 'subhodipsaha/molmoact2_pick_place'
CHUNK_SIZE = 30

print(f'Loading {REPO_ID}...')
t0 = time.time()
policy = MolmoAct2Policy.from_pretrained(REPO_ID)
policy = policy.to('cuda').eval().to(dtype=torch.bfloat16)
print(f'Loaded in {time.time()-t0:.1f}s  |  GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ── Load Molmo processor (lives in the base model repo only) ─────────────────
print('Loading Molmo processor from base model...')
proc = AutoProcessor.from_pretrained('allenai/MolmoAct2-SO100_101', trust_remote_code=True)
print(f'Processor: {type(proc).__name__}')

# ── Inject pick-place QUANTILE normalization as a new norm_tag ────────────────
_robot_stats = policy.model._get_robot_stats()
_NormCls     = type(next(iter(_robot_stats.state_normalizers.values())))

def _make_q01_q99_normalizer(q01, q99):
    n = object.__new__(_NormCls)
    n.mode      = 'q01_q99'
    n.mean      = None
    n.std       = None
    n.min_val   = np.array(q01, dtype=np.float32)
    n.max_val   = np.array(q99, dtype=np.float32)
    n.q_low     = np.array(q01, dtype=np.float32)
    n.q_high    = np.array(q99, dtype=np.float32)
    n.mask      = np.ones(len(q01), dtype=bool)
    n.zero_mask = np.zeros(len(q01), dtype=bool)
    return n

# Stats from policy_preprocessor_step_2_molmoact2_masked_normalizer.safetensors
_STATE_Q01  = [-23.165, -98.681,  22.969,  8.160, -99.811, 12.912]
_STATE_Q99  = [ 13.260,  10.139,  96.567, 76.626, -97.308, 58.226]
_ACTION_Q01 = [-23.262, -100.476, 19.592,  7.156, -100.066,  1.673]
_ACTION_Q99 = [ 13.886,   9.714,  97.071, 77.312,  -97.187, 58.590]

NORM_TAG = 'so101_pick_place'
_robot_stats.metadata_by_tag[NORM_TAG] = {'action_dim': 6}
_robot_stats.state_normalizers[NORM_TAG]  = _make_q01_q99_normalizer(_STATE_Q01, _STATE_Q99)
_robot_stats.action_normalizers[NORM_TAG] = _make_q01_q99_normalizer(_ACTION_Q01, _ACTION_Q99)

# Patch get_action_dim — the hf_model version ignores the metadata dict and
# returns None for custom tags; this patch makes it return 6 for our tag.
_orig_get_action_dim = _robot_stats.__class__.get_action_dim

def _patched_get_action_dim(self, norm_tag):
    if str(norm_tag) == NORM_TAG:
        return 6
    return _orig_get_action_dim(self, norm_tag)

_robot_stats.__class__.get_action_dim = _patched_get_action_dim
print(f'Injected norm_tag: {NORM_TAG!r}  |  action_dim check: {_robot_stats.get_action_dim(NORM_TAG)}')

# Training image dimensions
PHONE_SIZE = (480, 640)   # (W, H) for PIL.resize
WRIST_SIZE = (640, 480)   # (W, H) for PIL.resize

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK
Loading subhodipsaha/molmoact2_pick_place...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_rope_utils.py:1032: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(


Loading weights:   0%|          | 0/1295 [00:00<?, ?it/s]

Loaded in 314.5s  |  GPU mem: 11.0 GB
Loading Molmo processor from base model...


`MolmoAct2Processor` defines `image_processor_class = 'AutoImageProcessor'`, which is deprecated. Register the correct mapping in `AutoImageProcessor` instead.
`MolmoAct2Processor` defines `video_processor_class = 'AutoVideoProcessor'`, which is deprecated. Register the correct mapping in `AutoVideoProcessor` instead.


Processor: MolmoAct2Processor
Injected norm_tag: 'so101_pick_place'  |  action_dim check: 6


In [4]:
# ── Debug: test predict_action with injected QUANTILE norm_tag ───────────────
# Run after cell-load. If it prints "OK", the server should work.
import traceback as _tb
import numpy as np
from PIL import Image as _Img

_phone_pil = _Img.new('RGB', (480, 640))
_wrist_pil = _Img.new('RGB', (640, 480))
# Use HOME_POS as a realistic state test (not zeros)
_state = np.array([-7.7, -98.7, 96.5, 76.5, -98.4, 53.4], dtype=np.float32)

print(f'Testing predict_action with norm_tag={NORM_TAG!r} ...')
try:
    _out = policy.model.predict_action(
        processor=proc,
        images=[_phone_pil, _wrist_pil],
        task='Pick up the object and place it at the target location',
        state=_state,
        norm_tag=NORM_TAG,
        inference_action_mode='continuous',
        n_action_steps=CHUNK_SIZE,
        enable_cuda_graph=False,
    )
    _acts = _out.actions
    if _acts.ndim == 3:
        _acts = _acts[0]   # remove batch dim → [CHUNK_SIZE, 6]
    print(f'OK — shape: {_acts.shape}')
    print(f'First: {_acts[0].cpu().float().tolist()}')
    print(f'Last:  {_acts[-1].cpu().float().tolist()}')
except Exception:
    _tb.print_exc()

Testing predict_action with norm_tag='so101_pick_place' ...
OK — shape: torch.Size([30, 6])
First: [-7.46875, -100.0, 96.5, 77.5, -98.5, 53.25]
Last:  [-7.3125, -100.0, 96.5, 77.5, -98.5, 53.25]


In [5]:
import base64, io, time, os
import numpy as np
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import uvicorn

PORT = 8000
LOG  = '/content/molmoact2_server.log'
_logf = open(LOG, 'a', buffering=1)

def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    _logf.write(line + '\n')

app = FastAPI()

class PredictRequest(BaseModel):
    images_b64: List[str]   # [phone_b64, wrist_b64]
    state: List[float]       # 6 joint angles (raw, robot calibration convention)
    task: str

class PredictResponse(BaseModel):
    actions: List[List[float]]
    inference_ms: float

_server_ready = False   # True after first real prediction warms CUDA caches

@app.get('/health')
def health():
    return {'status': 'ok', 'gpu_mem_gb': round(torch.cuda.memory_allocated()/1e9, 2),
            'warmed_up': _server_ready}

@app.post('/predict', response_model=PredictResponse)
def predict(req: PredictRequest):
    global _server_ready
    t0 = time.time()

    phone_pil = Image.open(io.BytesIO(base64.b64decode(req.images_b64[0]))).convert('RGB')
    wrist_pil = Image.open(io.BytesIO(base64.b64decode(req.images_b64[1]))).convert('RGB')
    state_np  = np.array(req.state, dtype=np.float32)

    log(f"predict | state={[round(v,1) for v in req.state]}")

    # enable_cuda_graph=False: CUDA graphs freeze input buffers at capture time,
    # causing the model to silently replay warmup inputs for every subsequent call.
    output = policy.model.predict_action(
        processor=proc,
        images=[phone_pil, wrist_pil],
        task=req.task,
        state=state_np,
        norm_tag=NORM_TAG,
        inference_action_mode='continuous',
        n_action_steps=CHUNK_SIZE,
        enable_cuda_graph=False,
    )

    _server_ready = True
    actions = output.actions.cpu().float().numpy()
    if actions.ndim == 3:
        actions = actions[0]

    ms = (time.time() - t0) * 1000
    log(f"  → shape={actions.shape}  {ms:.0f}ms  first={np.round(actions[0], 2).tolist()}")
    return PredictResponse(actions=actions.tolist(), inference_ms=round(ms, 1))

# ── Restartable server lifecycle ──────────────────────────────────────────────
_uvicorn_server = None
_server_thread  = None

def start_server():
    global _uvicorn_server, _server_thread

    if _uvicorn_server is not None:
        log('Stopping previous server...')
        _uvicorn_server.should_exit = True
        _server_thread.join(timeout=5)
        time.sleep(0.5)

    try:
        ngrok.kill()
    except Exception:
        pass

    config = uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning')
    _uvicorn_server = uvicorn.Server(config)
    _server_thread  = threading.Thread(target=_uvicorn_server.run, daemon=True)
    _server_thread.start()
    time.sleep(2)

    ngrok.set_auth_token(os.environ['NGROK_TOKEN'])
    tunnel = ngrok.connect(PORT, 'http')
    log(f'Server ready | {tunnel.public_url}')
    print('\n' + '='*60)
    print(f'  SERVER URL: {tunnel.public_url}')
    print(f'  Monitor:    !tail -f {LOG}')
    print('  Paste URL into config.py as MOLMOACT2_SERVER')
    print('  NOTE: first /predict call warms CUDA caches (~1-2s); subsequent ~0.6-0.8s')
    print('='*60)
    return tunnel.public_url

start_server()

[06:34:26] Server ready | https://10eb-34-142-195-60.ngrok-free.app

  SERVER URL: https://10eb-34-142-195-60.ngrok-free.app
  Monitor:    !tail -f /content/molmoact2_server.log
  Paste URL into config.py as MOLMOACT2_SERVER
  NOTE: first /predict call warms CUDA caches (~1-2s); subsequent ~0.6-0.8s


'https://10eb-34-142-195-60.ngrok-free.app'

In [ ]:
# Run in a separate cell to tail live server logs
!tail -f /content/molmoact2_server.log

[06:21:48]   → shape=(30, 6)  310ms  first=[-5.059999942779541, -100.5, 97.0, 77.5, -98.0, 53.0]
[06:21:49] predict | state=[-5.1, -98.7, 98.2, 76.8, -98.2, 53.0] | graph=ready
[06:21:50]   → shape=(30, 6)  314ms  first=[-4.909999847412109, -100.5, 97.0, 77.5, -98.0, 53.0]
[06:21:51] predict | state=[-5.1, -98.7, 98.2, 76.8, -98.2, 53.0] | graph=ready
[06:21:52]   → shape=(30, 6)  312ms  first=[-4.880000114440918, -100.5, 97.0, 77.5, -98.0, 53.25]
[06:21:53] predict | state=[-5.1, -98.7, 98.2, 76.8, -98.1, 53.0] | graph=ready
[06:21:54]   → shape=(30, 6)  315ms  first=[-4.840000152587891, -100.5, 97.0, 77.5, -98.0, 53.25]
[06:21:55] predict | state=[-5.1, -98.7, 98.2, 76.8, -98.1, 53.0] | graph=ready
[06:21:56]   → shape=(30, 6)  310ms  first=[-4.840000152587891, -100.5, 97.0, 77.0, -98.0, 53.25]
[06:34:26] Server ready | https://10eb-34-142-195-60.ngrok-free.app
[06:34:57] predict | state=[-7.7, -98.7, 96.5, 76.5, -98.4, 53.4]
[06:34:57] predict | state=[-7.7, -98.7, 96.5, 76.5, -98.4